In [1]:
from utils import create_url, get_month_data
import os
import pandas as pd
from datetime import datetime, timedelta

# I. Weather Data

In [2]:
def fill_null_time(df):
    """
    Tự động điền các giá trị NULL trong cột 'time' bằng cách cộng 30 phút
    từ giá trị trước đó.
    df['time'] phải có dạng 'YYYY-MM-DDTHH:MM'
    """
    for i in df.index:
        if pd.isnull(df.at[i, "time"]):
            if i == 0:
                raise ValueError("Dòng đầu tiên NULL, không có giá trị trước để cộng 30 phút")
            prev_time = df.at[i-1, "time"]
            # Chuyển sang datetime và cộng 30 phút
            dt = datetime.strptime(prev_time, "%Y-%m-%dT%H:%M")
            dt_plus = dt + timedelta(minutes=30)
            df.at[i, "time"] = dt_plus.strftime("%Y-%m-%dT%H:%M")
    return df

In [3]:
station_code = 1904890

In [4]:
start_year, start_month = 2021, 10
end_year, end_month = 2021, 10
current_year = start_year
current_month = start_month

while (current_year < end_year) or (current_year == end_year and current_month <= end_month):
    print(f"Xử lý tháng {current_month}/{current_year}...")
 
    df = get_month_data(create_url(station_code, current_month, current_year)) # Lấy dữ liệu tháng
    df = fill_null_time(df) # Điền NULL cho cột time
    file_name = os.path.join(".", f"Data{current_month:02d}{current_year}.csv") # Lưu CSV
    df.to_csv(file_name, index=False)
    print(f"Đã lưu {file_name}")
    
    # Cập nhật tháng
    current_month += 1
    if current_month > 12:
        current_month = 1
        current_year += 1

Xử lý tháng 10/2021...
Đang chọn ngày: 1 October 2021
  - Lấy được 48 giờ, 48 weather, 48 wind, 48 RH, 48 P
Các giờ bị thiếu: []
Đang chọn ngày: 2 October 2021
  - Lấy được 47 giờ, 47 weather, 47 wind, 47 RH, 47 P
Các giờ bị thiếu: ['04:00']
Đang chọn ngày: 3 October 2021
  - Lấy được 47 giờ, 48 weather, 48 wind, 48 RH, 48 P
Các giờ bị thiếu: ['23:30']
Đang chọn ngày: 4 October 2021
  - Lấy được 48 giờ, 48 weather, 48 wind, 48 RH, 48 P
Các giờ bị thiếu: []
Đang chọn ngày: 5 October 2021
  - Lấy được 47 giờ, 47 weather, 47 wind, 47 RH, 47 P
Các giờ bị thiếu: ['05:00']
Đang chọn ngày: 6 October 2021
  - Lấy được 46 giờ, 46 weather, 46 wind, 46 RH, 46 P
Các giờ bị thiếu: ['07:00', '07:30']
Đang chọn ngày: 7 October 2021
  - Lấy được 43 giờ, 43 weather, 43 wind, 43 RH, 43 P
Các giờ bị thiếu: ['03:00', '04:00', '07:00', '10:30', '15:00']
Đang chọn ngày: 8 October 2021
  - Lấy được 43 giờ, 46 weather, 46 wind, 46 RH, 46 P
Các giờ bị thiếu: ['18:00', '20:00', '22:30', '23:00', '23:30']
Đang c

# Tổng hợp dữ liệu weather

In [5]:
def concat_data_files(start_month, start_year, end_month, end_year, folder="."):
    """
    Nối các file Data{month}{year}.csv từ start_month/start_year đến end_month/end_year
    và sắp xếp theo cột 'time'.
    
    start_month, end_month: int (1-12)
    start_year, end_year: int
    folder: thư mục chứa file
    """
    all_dfs = []
    
    # Tạo danh sách các tháng
    current_year = start_year
    current_month = start_month
    while (current_year < end_year) or (current_year == end_year and current_month <= end_month):
        month_str = f"{current_month:02d}"
        print(f"Đang đọc file cho tháng {month_str}/{current_year}...")
        file_name = os.path.join(folder, f"Data{month_str}{current_year}.csv")
        if os.path.exists(file_name):
            df = pd.read_csv(file_name)
            all_dfs.append(df)
        else:
            print(f"File {file_name} không tồn tại, bỏ qua.")
        
        # Cập nhật tháng
        current_month += 1
        if current_month > 12:
            current_month = 1
            current_year += 1
    
    # Nối tất cả các DataFrame
    if len(all_dfs) == 0:
        raise ValueError("Không có file nào được tìm thấy.")
    
    full_df = pd.concat(all_dfs, ignore_index=True)
    
    # Chuyển cột time sang datetime nếu chưa
    if not pd.api.types.is_datetime64_any_dtype(full_df['time']):
        full_df['time'] = pd.to_datetime(full_df['time'])
    
    # Sắp xếp theo time
    full_df = full_df.sort_values(by='time').reset_index(drop=True)
    
    return full_df

In [6]:
start_year, start_month = 2021, 9
end_year, end_month = 2025, 8

In [7]:
df_all = concat_data_files(start_month, start_year, end_month, end_year, folder=".")
df_all = df_all[df_all['time'].dt.minute == 0]
df_all.to_csv(os.path.join(".", f"Weather_ThuDuc_{start_month}{start_year}{end_month}{end_year}.csv"), index=False)

Đang đọc file cho tháng 09/2021...
Đang đọc file cho tháng 10/2021...
Đang đọc file cho tháng 11/2021...
Đang đọc file cho tháng 12/2021...
Đang đọc file cho tháng 01/2022...
Đang đọc file cho tháng 02/2022...
Đang đọc file cho tháng 03/2022...
Đang đọc file cho tháng 04/2022...
Đang đọc file cho tháng 05/2022...
Đang đọc file cho tháng 06/2022...
Đang đọc file cho tháng 07/2022...
Đang đọc file cho tháng 08/2022...
Đang đọc file cho tháng 09/2022...
Đang đọc file cho tháng 10/2022...
Đang đọc file cho tháng 11/2022...
Đang đọc file cho tháng 12/2022...
Đang đọc file cho tháng 01/2023...
Đang đọc file cho tháng 02/2023...
Đang đọc file cho tháng 03/2023...
Đang đọc file cho tháng 04/2023...
Đang đọc file cho tháng 05/2023...
Đang đọc file cho tháng 06/2023...
Đang đọc file cho tháng 07/2023...
Đang đọc file cho tháng 08/2023...
Đang đọc file cho tháng 09/2023...
Đang đọc file cho tháng 10/2023...
Đang đọc file cho tháng 11/2023...
Đang đọc file cho tháng 12/2023...
Đang đọc file cho th

In [8]:
df_all

,time,temp,weather,wind,RH,P
0,2023-01-01 00:00:00,24.0,Passing clouds.,6.0,69.0,1014.0
2,2023-01-01 01:00:00,24.0,Passing clouds.,9.0,69.0,1013.0
4,2023-01-01 02:00:00,24.0,Passing clouds.,6.0,69.0,1013.0
6,2023-01-01 03:00:00,24.0,Passing clouds.,6.0,69.0,1013.0
8,2023-01-01 04:00:00,23.0,Passing clouds.,9.0,74.0,1012.0
...,...,...,...,...,...,...
43365,2025-08-31 19:00:00,26.0,Light rain. Passing clouds.,6.0,100.0,1006.0
43367,2025-08-31 20:00:00,27.0,Passing clouds.,9.0,94.0,1007.0
43369,2025-08-31 21:00:00,27.0,Passing clouds.,13.0,94.0,1007.0
43371,2025-08-31 22:00:00,26.0,Clear.,11.0,100.0,1008.0


# II. Chemical Data

In [ ]:
import requests
import time
import pandas as pd

lat = 10.85
lon = 106.75

# Thời gian bắt đầu và kết thúc
start = int(time.mktime(time.strptime('2021-09-01 00:00:00', '%Y-%m-%d %H:%M:%S')))
end   = int(time.mktime(time.strptime('2022-12-31 23:00:00', '%Y-%m-%d %H:%M:%S')))

# API key của bạn
api_key = "12203f83360b3cf04178716b52f39419"

# URL API
url = f"http://api.openweathermap.org/data/2.5/air_pollution/history?lat={lat}&lon={lon}&start={start}&end={end}&appid={api_key}"

# Gửi yêu cầu GET
response = requests.get(url)

data = response.json()
df = pd.DataFrame(data['list'])
df['datetime'] = pd.to_datetime(df['dt'], unit='s')

In [7]:
df

,main,components,dt,datetime
0,{'aqi': 2},"{'co': 607.49, 'no': 4.86, 'no2': 20.91, 'o3':...",1627750800,2021-07-31 17:00:00
1,{'aqi': 2},"{'co': 534.06, 'no': 3.13, 'no2': 20.39, 'o3':...",1627754400,2021-07-31 18:00:00
2,{'aqi': 2},"{'co': 487.33, 'no': 2.4, 'no2': 19.71, 'o3': ...",1627758000,2021-07-31 19:00:00
3,{'aqi': 2},"{'co': 460.63, 'no': 1.98, 'no2': 19.19, 'o3':...",1627761600,2021-07-31 20:00:00
4,{'aqi': 2},"{'co': 447.27, 'no': 1.82, 'no2': 19.19, 'o3':...",1627765200,2021-07-31 21:00:00
...,...,...,...,...
12259,{'aqi': 4},"{'co': 2162.93, 'no': 0.36, 'no2': 63.75, 'o3'...",1672488000,2022-12-31 12:00:00
12260,{'aqi': 4},"{'co': 1976.01, 'no': 1.13, 'no2': 56.21, 'o3'...",1672491600,2022-12-31 13:00:00
12261,{'aqi': 4},"{'co': 1615.52, 'no': 0.92, 'no2': 44.55, 'o3'...",1672495200,2022-12-31 14:00:00
12262,{'aqi': 4},"{'co': 1321.79, 'no': 0.54, 'no2': 35.99, 'o3'...",1672498800,2022-12-31 15:00:00


In [8]:
import pandas as pd

# Lấy danh sách các chemical từ hàng đầu tiên
col = list(df['components'][0].keys())

# Khởi tạo list để chứa dict từng hàng
rows = []

for i in range(len(df)):
    row = {}
    utc_time = pd.to_datetime(df.loc[i, 'dt'], unit='s', utc=True)
    vn_time = utc_time.tz_convert("Asia/Ho_Chi_Minh").tz_localize(None)
    row['time'] = vn_time  # lấy datetime
    for chemical in col:
        row[chemical] = df.loc[i, 'components'][chemical]
    rows.append(row)

# Tạo DataFrame từ list of dicts
data_using = pd.DataFrame(rows)

In [9]:
data_using

,time,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2021-08-01 00:00:00,607.49,4.86,20.91,0.65,19.79,15.06,21.08,4.31
1,2021-08-01 01:00:00,534.06,3.13,20.39,1.35,18.84,12.82,17.70,3.86
2,2021-08-01 02:00:00,487.33,2.40,19.71,2.10,18.60,11.75,16.02,3.77
3,2021-08-01 03:00:00,460.63,1.98,19.19,2.77,18.60,10.92,14.71,3.64
4,2021-08-01 04:00:00,447.27,1.82,19.19,3.22,18.84,10.26,13.72,3.58
...,...,...,...,...,...,...,...,...,...
12259,2022-12-31 19:00:00,2162.93,0.36,63.75,15.20,29.09,62.41,89.70,18.75
12260,2022-12-31 20:00:00,1976.01,1.13,56.21,6.79,24.32,62.82,96.60,20.01
12261,2022-12-31 21:00:00,1615.52,0.92,44.55,7.15,18.60,57.70,93.28,19.00
12262,2022-12-31 22:00:00,1321.79,0.54,35.99,9.39,15.26,51.14,85.26,15.33


In [ ]:
start_month = "9"
start_year = "2021"
end_month = "8"
end_year = "2025"

In [13]:
data_using.to_csv(os.path.join(".", f"Chemical_ThuDuc_{start_month}{start_year}{end_month}{end_year}.csv"), index=False)

In [26]:
df_merged.head(5)

,time,co,no,no2,o3,so2,pm2_5,pm10,nh3
0,2021-08-01 00:00:00,607.49,4.86,20.91,0.65,19.79,15.06,21.08,4.31
1,2021-08-01 01:00:00,534.06,3.13,20.39,1.35,18.84,12.82,17.70,3.86
2,2021-08-01 02:00:00,487.33,2.40,19.71,2.10,18.60,11.75,16.02,3.77
3,2021-08-01 03:00:00,460.63,1.98,19.19,2.77,18.60,10.92,14.71,3.64
4,2021-08-01 04:00:00,447.27,1.82,19.19,3.22,18.84,10.26,13.72,3.58


In [27]:
df_merged.tail(5)

,time,co,no,no2,o3,so2,pm2_5,pm10,nh3
35107,2025-08-31 19:00:00,155.68,0.01,3.11,19.38,0.78,1.63,2.68,10.42
35108,2025-08-31 20:00:00,162.19,0.01,3.37,18.34,0.83,1.97,3.29,11.15
35109,2025-08-31 21:00:00,169.83,0.01,3.61,17.46,0.87,2.37,3.97,11.93
35110,2025-08-31 22:00:00,175.91,0.01,3.81,16.92,0.90,2.75,4.66,12.43
35111,2025-08-31 23:00:00,178.16,0.01,3.91,16.48,0.89,3.02,5.17,12.60


In [30]:
df_merged.to_csv("Chemical_ThuDuc_082021082025.csv", index=False)

# III. Merge

In [8]:
start_month = "9"
start_year = "2021"
end_month = "8"
end_year = "2025"

In [9]:
import pandas as pd
import os
weather_df = pd.read_csv(os.path.join(".", f"Weather_ThuDuc_{start_month}{start_year}{end_month}{end_year}.csv"))
chemical_df = pd.read_csv(os.path.join(".", f"Chemical_ThuDuc_{start_month}{start_year}{end_month}{end_year}.csv"))
merged_df = pd.merge(weather_df, chemical_df, on="time", how="outer")

In [10]:
merged_df.to_csv(os.path.join(".", f"Dataset_ThuDuc_{start_month}{start_year}{end_month}{end_year}.csv"), index=False)

# IV. Chia train, val và test theo tỉ lệ 7:1.5:1.5

In [11]:
start_year, start_month = 2021, 9
end_year, end_month = 2025, 8

In [12]:
data = pd.read_csv(os.path.join(".", f"Dataset_ThuDuc_{start_month}{start_year}{end_month}{end_year}.csv"))
data["time"] = pd.to_datetime(data["time"])
train = data[data["time"] < pd.to_datetime("2024-09-01")]
val   = data[(data["time"] >= pd.to_datetime("2024-09-01")) & (data["time"] < pd.to_datetime("2025-03-01"))]
test  = data[data["time"] >= pd.to_datetime("2025-03-01")]

In [13]:
print(train.shape)
print(val.shape)
print(test.shape)

(26280, 14)
(4352, 14)
(4422, 14)
